# 03. Como o modelo usa funções Python

O modelo não acessa o JSON por conta própria. Vamos oferecer ferramentas para consultar os dados. O modelo solicita uma função e informa os argumentos; nosso código executa a função e devolve o resultado. Comece carregando a base e listando os temas.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from science_tutor.repository import KnowledgeBase
kb = KnowledgeBase(ROOT / "data" / "knowledge_base.json")
kb.list_topics()

[{'id': 'plate_tectonics',
  'name': 'Plate Tectonics',
  'category': 'earth_science',
  'description': 'Interactions among lithospheric plates.'},
 {'id': 'water_cycle',
  'name': 'Water Cycle',
  'category': 'earth_science',
  'description': 'Movement of water through Earth systems.'},
 {'id': 'erosion',
  'name': 'Erosion and Deposition',
  'category': 'earth_science',
  'description': 'Weathering, transport, erosion, and deposition.'},
 {'id': 'cells',
  'name': 'Cells',
  'category': 'life_science',
  'description': 'Cell structures and transport across membranes.'},
 {'id': 'ecosystems',
  'name': 'Ecosystems',
  'category': 'life_science',
  'description': 'Energy flow and interactions in ecosystems.'},
 {'id': 'forces_motion',
  'name': 'Forces and Motion',
  'category': 'physical_science',
  'description': 'Net force, balanced forces, and motion.'}]

## Desafio 3: escreva uma função comum

Crie uma função que retorne os temas usando `kb.list_topics()`. Execute-a e confira o resultado. Por enquanto, ela é apenas uma função Python: ainda não foi apresentada ao modelo como ferramenta.

In [2]:
def my_list_topics():
    return kb.list_topics()

my_list_topics()

[{'id': 'plate_tectonics',
  'name': 'Plate Tectonics',
  'category': 'earth_science',
  'description': 'Interactions among lithospheric plates.'},
 {'id': 'water_cycle',
  'name': 'Water Cycle',
  'category': 'earth_science',
  'description': 'Movement of water through Earth systems.'},
 {'id': 'erosion',
  'name': 'Erosion and Deposition',
  'category': 'earth_science',
  'description': 'Weathering, transport, erosion, and deposition.'},
 {'id': 'cells',
  'name': 'Cells',
  'category': 'life_science',
  'description': 'Cell structures and transport across membranes.'},
 {'id': 'ecosystems',
  'name': 'Ecosystems',
  'category': 'life_science',
  'description': 'Energy flow and interactions in ecosystems.'},
 {'id': 'forces_motion',
  'name': 'Forces and Motion',
  'category': 'physical_science',
  'description': 'Net force, balanced forces, and motion.'}]

## Conheça as ferramentas do projeto

O decorador `@tool` transforma uma função em uma ferramenta que podemos apresentar ao modelo. Leia os nomes e as descrições exibidos abaixo. As descrições explicam quando usar cada função.

Depois, `bind_tools` informa ao modelo quais ferramentas estão disponíveis. Confira `response.tool_calls`: essa lista contém os pedidos de execução, não os resultados. Na célula seguinte, executamos o primeiro pedido. Se a lista estiver vazia, reformule a pergunta antes de continuar.

In [3]:
from science_tutor.tools import TOOLS
for t in TOOLS:
    print(t.name, "->", t.description)

list_topics -> Lista todos os tópicos de ciências disponíveis.
search_questions -> Busca perguntas de um tópico. Use antes de escolher um question_id.
get_question -> Carrega uma pergunta (com alternativas), mas NUNCA a resposta.
check_answer -> Verifica a resposta do aluno. Aceita 'A', 'a)', ou o texto da opção.


In [5]:
from langchain_ollama import ChatOllama
model = ChatOllama(base_url="http://127.0.0.1:11434", model="qwen3.5:4b", temperature=0).bind_tools(TOOLS)
response = model.invoke("Quais temas de ciências posso estudar com este tutor?")
print("Resposta:", response)
print("Chamadas de ferramentas:", response)

Resposta: content='' additional_kwargs={} response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-10-28T17:34:15.766972609Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1764009599, 'load_duration': 714221, 'prompt_eval_count': 486, 'prompt_eval_duration': 557334000, 'eval_count': 56, 'eval_duration': 1200696000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'} id='lc_run--01a18393-a5e6-75d3-b780-7e70b33b6306-0' tool_calls=[{'name': 'list_topics', 'args': {}, 'id': '9ceffa46-e817-417d-ac10-a2a6b7e07967', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 486, 'output_tokens': 56, 'total_tokens': 542}
Chamadas de ferramentas: content='' additional_kwargs={} response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-10-28T17:34:15.766972609Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1764009599, 'load_duration': 714221, 'prompt_eval_count': 486, 'prompt_eval_duration': 557334000, 'eval_count': 56, 'eval_d

In [ ]:
tool_map = {tool.name: tool for tool in TOOLS}
call = response.tool_calls[0]
result = tool_map[call["name"]].invoke(call["args"])
print(result)

## Devolva o resultado ao modelo

A função já foi executada, mas o modelo ainda não recebeu seu resultado. Envie uma `ToolMessage` com o resultado e o `tool_call_id` do pedido correspondente. Observe a sequência: pergunta, pedido de ferramenta, resultado e resposta final.

In [ ]:
from langchain.messages import HumanMessage, ToolMessage

history = [
    HumanMessage(content="Quais temas de ciências posso estudar com este tutor? Responda em português."),
    response,
    ToolMessage(content=result, tool_call_id=call["id"]),
]

final = model.invoke(history)
print(final.content)


## Desafio 4: teste outras ferramentas

Faça perguntas que usem `search_questions`, `get_question` e `check_answer`. Use um identificador de pergunta retornado pela base. Para cada teste:

1. Confira o nome e os argumentos em `tool_calls`.
2. Execute a ferramenta solicitada.
3. Devolva o resultado em uma `ToolMessage`.
4. Chame o modelo novamente para obter a resposta ao usuário.

No próximo notebook, vamos automatizar esse ciclo com LangGraph.